In [1]:
pip install nltk gensim

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
# LAB ASSIGNMENT 3 : BAG OF WORDS (BoW) AND N-GRAM MODEL
# PART A - BAG OF WORDS

doc1 = "Cats and dogs are not allowed"
doc2 = "Cats and dogs are antagonistic"

corpus = [doc1, doc2]
print("Document 1:", doc1)
print("Document 2:", doc2)

Document 1: Cats and dogs are not allowed
Document 2: Cats and dogs are antagonistic


In [3]:
tokenized_docs = [doc.lower().split() for doc in corpus]

for i, tokens in enumerate(tokenized_docs, 1):
    print(f"Document {i} tokens: {tokens}")

Document 1 tokens: ['cats', 'and', 'dogs', 'are', 'not', 'allowed']
Document 2 tokens: ['cats', 'and', 'dogs', 'are', 'antagonistic']


In [4]:
# Bag of Words first makes a unique list of all the words in the documents
vocabulary = []
for tokens in tokenized_docs:
    for word in tokens:
        if word not in vocabulary:
            vocabulary.append(word)

print("Unique words (vocabulary):", vocabulary)
print("Vocabulary size:", len(vocabulary))

Unique words (vocabulary): ['cats', 'and', 'dogs', 'are', 'not', 'allowed', 'antagonistic']
Vocabulary size: 7


In [5]:
# Each document becomes a feature vector.
# The words in the document are marked as 1 and the remaining as 0.
binary_vectors = []
for tokens in tokenized_docs:
    vector = [1 if word in tokens else 0 for word in vocabulary]
    binary_vectors.append(vector)

print("Vocabulary      :", vocabulary)
print("Document1 vector:", binary_vectors[0])
print("Document2 vector:", binary_vectors[1])

Vocabulary      : ['cats', 'and', 'dogs', 'are', 'not', 'allowed', 'antagonistic']
Document1 vector: [1, 1, 1, 1, 1, 1, 0]
Document2 vector: [1, 1, 1, 1, 0, 0, 1]


In [6]:
# The BoW model can also keep the frequency of each word
count_vectors = []
for tokens in tokenized_docs:
    vector = [tokens.count(word) for word in vocabulary]
    count_vectors.append(vector)

print("Document1 counts:", count_vectors[0])
print("Document2 counts:", count_vectors[1])

Document1 counts: [1, 1, 1, 1, 1, 1, 0]
Document2 counts: [1, 1, 1, 1, 0, 0, 1]


In [7]:
import pandas as pd

bow_df = pd.DataFrame(count_vectors, columns=vocabulary, index=["Doc1", "Doc2"])
print("--- BAG OF WORDS MATRIX ---")
bow_df

--- BAG OF WORDS MATRIX ---


,cats,and,dogs,are,not,allowed,antagonistic
Doc1,1,1,1,1,1,1,0
Doc2,1,1,1,1,0,0,1


In [8]:
# Second sample input : "It was the best of times, It was the worst of times."
lines = ["It was the best of times", "It was the worst of times"]
line_tokens = [line.lower().split() for line in lines]

vocab2 = []
for tokens in line_tokens:
    for word in tokens:
        if word not in vocab2:
            vocab2.append(word)

print("Vocabulary:", vocab2)
for i, tokens in enumerate(line_tokens, 1):
    print(f"Vector {i}:", [1 if w in tokens else 0 for w in vocab2])

Vocabulary: ['it', 'was', 'the', 'best', 'of', 'times', 'worst']
Vector 1: [1, 1, 1, 1, 1, 1, 0]
Vector 2: [1, 1, 1, 0, 1, 1, 1]


In [9]:
# PART B - BAG OF WORDS WITH GENSIM

import gensim
import pprint
from gensim import corpora
from gensim.utils import simple_preprocess

doc_list = [
    "Hello, how are you?", "How do you do?",
    "Hey what are you doing? yes you What are you doing?"
]
print(doc_list)

['Hello, how are you?', 'How do you do?', 'Hey what are you doing? yes you What are you doing?']


In [10]:
doc_tokenized = [simple_preprocess(doc) for doc in doc_list]
print(doc_tokenized)

[['hello', 'how', 'are', 'you'], ['how', 'do', 'you', 'do'], ['hey', 'what', 'are', 'you', 'doing', 'yes', 'you', 'what', 'are', 'you', 'doing']]


In [11]:
# The Dictionary object maps every word to a unique integer id
dictionary = corpora.Dictionary()
BoW_corpus = [dictionary.doc2bow(doc, allow_update=True) for doc in doc_tokenized]
print(BoW_corpus)

[[(0, 1), (1, 1), (2, 1), (3, 1)], [(2, 1), (3, 1), (4, 2)], [(0, 2), (3, 3), (5, 2), (6, 1), (7, 2), (8, 1)]]


In [12]:
print("--- DICTIONARY : WORD TO INTEGER ID ---")
pprint.pprint(dictionary.token2id)

--- DICTIONARY : WORD TO INTEGER ID ---
{'are': 0,
 'do': 4,
 'doing': 5,
 'hello': 1,
 'hey': 6,
 'how': 2,
 'what': 7,
 'yes': 8,
 'you': 3}


In [13]:
# doc2bow returns a sparse vector of (word_id, count) pairs.
# Convert the ids back to words to read the result.
readable_corpus = [[(dictionary[word_id], count) for word_id, count in doc]
                   for doc in BoW_corpus]
pprint.pprint(readable_corpus)

[[('are', 1), ('hello', 1), ('how', 1), ('you', 1)],
 [('how', 1), ('you', 1), ('do', 2)],
 [('are', 2), ('you', 3), ('doing', 2), ('hey', 1), ('what', 2), ('yes', 1)]]


In [14]:
# The model only knows the words in the dictionary.
# Set allow_update=False to see that new words are discarded.
new_doc = simple_preprocess("How are you today my friend")
print("New document tokens:", new_doc)

new_bow = dictionary.doc2bow(new_doc, allow_update=False)
print("BoW vector:", [(dictionary[i], count) for i, count in new_bow])
print("Unknown words dropped:", [w for w in new_doc if w not in dictionary.token2id])

New document tokens: ['how', 'are', 'you', 'today', 'my', 'friend']
BoW vector: [('are', 1), ('how', 1), ('you', 1)]
Unknown words dropped: ['today', 'my', 'friend']


In [15]:
# LIMITATIONS OF BAG OF WORDS
# 1. The model ignores the location information of the word.

def make_bow(sentences):
    tokens = [s.lower().split() for s in sentences]
    vocab = sorted({w for t in tokens for w in t})
    vectors = [[t.count(w) for w in vocab] for t in tokens]
    return vocab, vectors

vocab_order, vectors_order = make_bow(["today is off", "is today off"])

print("Vocabulary     :", vocab_order)
print("'today is off' :", vectors_order[0])
print("'is today off' :", vectors_order[1])
print("Vectors are equal:", vectors_order[0] == vectors_order[1])

Vocabulary     : ['is', 'off', 'today']
'today is off' : [1, 1, 1]
'is today off' : [1, 1, 1]
Vectors are equal: True


In [16]:
# 2. The model does not respect the semantics of the word.
vocab_sem, vectors_sem = make_bow(["I play soccer every weekend",
                                   "I play football every weekend"])

print("Vocabulary        :", vocab_sem)
print("Soccer sentence   :", vectors_sem[0])
print("Football sentence :", vectors_sem[1])
print("'soccer' and 'football' get different columns although they mean the same.")

Vocabulary        : ['every', 'football', 'i', 'play', 'soccer', 'weekend']
Soccer sentence   : [1, 0, 1, 1, 1, 1]
Football sentence : [1, 1, 1, 1, 0, 1]
'soccer' and 'football' get different columns although they mean the same.


In [17]:
# PART C - N-GRAM MODEL

from nltk.util import ngrams

sentence = "This is a sentence"
tokens = sentence.lower().split()

print("Sentence:", sentence)
for n, name in [(1, "unigrams"), (2, "bigrams"), (3, "trigrams")]:
    grams = list(ngrams(tokens, n))
    print(f"N = {n} {name:<9}: {grams}")

Sentence: This is a sentence
N = 1 unigrams : [('this',), ('is',), ('a',), ('sentence',)]
N = 2 bigrams  : [('this', 'is'), ('is', 'a'), ('a', 'sentence')]
N = 3 trigrams : [('this', 'is', 'a'), ('is', 'a', 'sentence')]


In [18]:
from nltk.util import bigrams, trigrams

words = ["natural", "language", "processing", "is", "fun"]

print("Unigrams:", words)
print("Bigrams :", list(bigrams(words)))
print("Trigrams:", list(trigrams(words)))

Unigrams: ['natural', 'language', 'processing', 'is', 'fun']
Bigrams : [('natural', 'language'), ('language', 'processing'), ('processing', 'is'), ('is', 'fun')]
Trigrams: [('natural', 'language', 'processing'), ('language', 'processing', 'is'), ('processing', 'is', 'fun')]


In [19]:
# An n-gram model is built by counting how often the word sequences
# occur in the corpus text.
from collections import Counter

train_sentences = [
    "I love natural language processing",
    "I love machine learning",
    "I study natural language processing",
    "she loves natural language processing",
    "I love deep learning models"
]

train_tokens = [s.lower().split() for s in train_sentences]
for t in train_tokens:
    print(t)

['i', 'love', 'natural', 'language', 'processing']
['i', 'love', 'machine', 'learning']
['i', 'study', 'natural', 'language', 'processing']
['she', 'loves', 'natural', 'language', 'processing']
['i', 'love', 'deep', 'learning', 'models']


In [20]:
unigram_counts = Counter()
bigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(tokens)
    bigram_counts.update(list(bigrams(tokens)))

print("--- TOP 5 UNIGRAMS ---")
print(unigram_counts.most_common(5))
print("--- TOP 5 BIGRAMS ---")
print(bigram_counts.most_common(5))

--- TOP 5 UNIGRAMS ---
[('i', 4), ('love', 3), ('natural', 3), ('language', 3), ('processing', 3)]
--- TOP 5 BIGRAMS ---
[(('i', 'love'), 3), (('natural', 'language'), 3), (('language', 'processing'), 3), (('love', 'natural'), 1), (('love', 'machine'), 1)]


In [21]:
# The counts give the probability P(w|h) : the chance of word w
# after history h.
def bigram_probability(history, word):
    if unigram_counts[history] == 0:
        return 0.0
    return bigram_counts[(history, word)] / unigram_counts[history]

print(f"{'BIGRAM':<28} {'COUNT':<7} {'P(w|h)'}")
print("-" * 45)
for (h, w), count in bigram_counts.most_common(8):
    print(f"{h + ' -> ' + w:<28} {count:<7} {bigram_probability(h, w):.3f}")

BIGRAM                       COUNT   P(w|h)
---------------------------------------------
i -> love                    3       0.750
natural -> language          3       1.000
language -> processing       3       1.000
love -> natural              1       0.333
love -> machine              1       0.333
machine -> learning          1       1.000
i -> study                   1       0.250
study -> natural             1       1.000


In [22]:
# A good n-gram model predicts the next word in the sentence
def predict_next_word(history, top_k=3):
    candidates = {w: bigram_probability(history, w)
                  for (h, w) in bigram_counts if h == history}
    return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:top_k]

for history in ["i", "love", "natural"]:
    print(f"After '{history}':", predict_next_word(history))

After 'i': [('love', 0.75), ('study', 0.25)]
After 'love': [('natural', 0.3333333333333333), ('machine', 0.3333333333333333), ('deep', 0.3333333333333333)]
After 'natural': [('language', 1.0)]


In [23]:
# The probability of a full sentence is the product of the bigram
# probabilities.
def sentence_probability(sentence):
    tokens = sentence.lower().split()
    prob = 1.0
    print(f"Sentence: '{sentence}'")
    for h, w in bigrams(tokens):
        p = bigram_probability(h, w)
        print(f"   P('{w}' | '{h}') = {p:.3f}")
        prob *= p
    print(f"   TOTAL PROBABILITY = {prob:.5f}\n")

sentence_probability("I love natural language processing")
sentence_probability("I love quantum physics")

Sentence: 'I love natural language processing'
   P('love' | 'i') = 0.750
   P('natural' | 'love') = 0.333
   P('language' | 'natural') = 1.000
   P('processing' | 'language') = 1.000
   TOTAL PROBABILITY = 0.25000

Sentence: 'I love quantum physics'
   P('love' | 'i') = 0.750
   P('quantum' | 'love') = 0.000
   P('physics' | 'quantum') = 0.000
   TOTAL PROBABILITY = 0.00000



In [24]:
# The n-gram model keeps the word order that the BoW model loses
pair = ["today is off", "is today off"]
print("BoW vectors    :", vectors_order[0], "and", vectors_order[1], "-> same")
for s in pair:
    print(f"Bigrams of '{s}':", list(bigrams(s.split())))

BoW vectors    : [1, 1, 1] and [1, 1, 1] -> same
Bigrams of 'today is off': [('today', 'is'), ('is', 'off')]
Bigrams of 'is today off': [('is', 'today'), ('today', 'off')]


In [25]:
# SUMMARY

print("=" * 60)
print("BAG OF WORDS vs N-GRAM MODEL")
print("=" * 60)
print(f"{'PROPERTY':<24} {'BAG OF WORDS':<18} {'N-GRAM'}")
print("-" * 60)
rows = [
    ("Keeps word order",   "No",             "Yes, inside the gram"),
    ("Output",             "Fixed vector",   "Probability P(w|h)"),
    ("Built by",           "Word counts",    "Sequence counts"),
    ("Main use",           "Text features",  "Next word prediction"),
]
for r in rows:
    print(f"{r[0]:<24} {r[1]:<18} {r[2]}")
print("-" * 60)
print("BoW answers  : which words are in the document?")
print("N-gram answers: which word comes next?")

BAG OF WORDS vs N-GRAM MODEL
PROPERTY                 BAG OF WORDS       N-GRAM
------------------------------------------------------------
Keeps word order         No                 Yes, inside the gram
Output                   Fixed vector       Probability P(w|h)
Built by                 Word counts        Sequence counts
Main use                 Text features      Next word prediction
------------------------------------------------------------
BoW answers  : which words are in the document?
N-gram answers: which word comes next?
